
## Notebook 01: Dataset Exploration and Schema Verification


**Dataset:** SR-BH 2020 Multi-Label Web Attack Dataset (`data_capec_multilabel.csv`)


1. Load the raw multi-label HTTP request dataset.
2. Inspect the dataset structure, schema, and column types.
3. Classify attributes into detection features, contextual correlation features, response features, and target attack labels.
4. Check for duplicate records and missing data in core request attributes.
5. Analyze class distributions, severe class imbalance, and multi-label characteristics.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to python path for modular imports
current_dir = Path.cwd()
if str(current_dir.parent) not in sys.path:
    sys.path.append(str(current_dir.parent))

from src.config import (
    RAW_DATA_FILE,
    CORE_COLUMNS,
    CONTEXT_COLUMNS,
    DETECTION_FEATURES,
    TEXT_FEATURES,
    CATEGORICAL_FEATURES,
    RESPONSE_FEATURES,
    SPARSE_EXCLUDED_FEATURES,
    ALL_LABEL_COLUMNS,
    RARE_LABEL,
    MODEL_LABEL_COLUMNS
)

print("Environment initialized successfully.")
print("Raw dataset path:", RAW_DATA_FILE)

Environment initialized successfully.
Raw dataset path: C:\Users\Meishaaa\OneDrive - Sri Lanka Institute of Information Technology\Desktop updates\Desktop\RA 1\Dataset\data_capec_multilabel.csv


## 1. Load Raw Dataset

We load the complete SR-BH 2020 dataset from the `Dataset/` directory. `low_memory=False` is used to cleanly handle mixed-type HTTP header columns.

In [2]:
# Load the dataset
df = pd.read_csv(RAW_DATA_FILE, low_memory=False)

print("Dataset loaded successfully.")
print(f"Total records (rows): {df.shape[0]:,}")
print(f"Total attributes (columns): {df.shape[1]}")

Dataset loaded successfully.
Total records (rows): 907,815
Total attributes (columns): 38


## 2. Dataset Schema & Feature Taxonomy

The dataset consists of 38 columns, which serve distinct roles across the two research stages:
- **Contextual Attributes (5 fields):** `timestamp`, `src_ip`, `src_port`, `dst_ip`, `dst_port`. Kept strictly for Stage 2 (Event Correlation in PP2).
- **Request-Side Detection Features (14 fields):** 2 text fields (`request_http_request`, `request_body`) and 12 categorical fields. Inputs for Stage 1 Detection Model in PP1.
- **Excluded Fields (5 fields):** 4 server-side response fields (which would cause data leakage if used before request execution) and 1 highly sparse field (`request_do_not_track`).
- **CAPEC Target Labels (14 multi-labels):** Ground-truth attack categories based on the CAPEC taxonomy.

In [3]:
print(f"Request Text Features ({len(TEXT_FEATURES)}):")
for f in TEXT_FEATURES:
    print(f"  - {f}")

print(f"\nRequest Categorical Features ({len(CATEGORICAL_FEATURES)}):")
for f in CATEGORICAL_FEATURES:
    print(f"  - {f}")

print(f"\nPreserved Contextual Attributes ({len(CONTEXT_COLUMNS)}):")
for f in CONTEXT_COLUMNS:
    print(f"  - {f}")

print(f"\nExcluded Server-Side Response Features ({len(RESPONSE_FEATURES)}):")
for f in RESPONSE_FEATURES:
    print(f"  - {f}")

print(f"\nTarget CAPEC Multi-Labels ({len(ALL_LABEL_COLUMNS)}):")
for l in ALL_LABEL_COLUMNS:
    print(f"  - {l}")

Request Text Features (2):
  - request_http_request
  - request_body

Request Categorical Features (12):
  - request_http_method
  - request_http_protocol
  - request_host
  - request_user_agent
  - request_referer
  - request_origin
  - request_cookie
  - request_content_type
  - request_accept
  - request_accept_language
  - request_accept_encoding
  - request_connection

Preserved Contextual Attributes (5):
  - timestamp
  - src_ip
  - src_port
  - dst_ip
  - dst_port

Excluded Server-Side Response Features (4):
  - response_http_protocol
  - response_http_status_code
  - response_http_status_message
  - response_content_length

Target CAPEC Multi-Labels (14):
  - 000 - Normal
  - 272 - Protocol Manipulation
  - 242 - Code Injection
  - 88 - OS Command Injection
  - 126 - Path Traversal
  - 66 - SQL Injection
  - 16 - Dictionary-based Password Attack
  - 310 - Scanning for Vulnerable Software
  - 153 - Input Data Manipulation
  - 248 - Command Injection
  - 274 - HTTP Verb Tampering

## 3. Duplicate Records Verification

Exact duplicate records occur when identical HTTP requests are repeated. In machine learning, keeping exact duplicate rows can cause data leakage if copies appear across both train and test splits, and artificially inflates performance.

In [4]:
duplicate_count = df.duplicated().sum()
print(f"Exact duplicate rows found: {duplicate_count:,} ({(duplicate_count/len(df))*100:.2f}%)")
print(f"Expected rows remaining after deduplication: {len(df) - duplicate_count:,}")

Exact duplicate rows found: 2,746 (0.30%)
Expected rows remaining after deduplication: 905,069


## 4. Missing Core HTTP Records & Feature Nullity

Certain fields are fundamental to an HTTP request (such as method, protocol, request URI, IP addresses). If these are missing, the record is malformed and cannot be processed reliably.

In [5]:
# Check rows missing core attributes
core_missing = df[df[CORE_COLUMNS].isnull().any(axis=1)]
print(f"Rows missing core HTTP request information: {len(core_missing)}")
if len(core_missing) > 0:
    display(core_missing[CORE_COLUMNS])

Rows missing core HTTP request information: 2


,timestamp,src_ip,src_port,dst_ip,dst_port,request_http_method,request_http_request,request_http_protocol
14410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Missing Values across All Features

In HTTP logs, optional headers (like `request_referer`, `request_cookie`, `request_body`) are often absent in normal benign requests. This does not indicate corrupted records, but rather standard HTTP protocol behavior where headers are optional.

In [6]:
missing_counts = df.isnull().sum()
missing_df = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing (%)": (missing_counts / len(df)) * 100
}).sort_values("Missing Count", ascending=False)

display(missing_df[missing_df["Missing Count"] > 0])

,Missing Count,Missing (%)
request_do_not_track,907110,99.922341
request_origin,906206,99.822761
request_accept_language,891955,98.252948
request_body,881072,97.054135
request_content_type,879271,96.855747
request_referer,616421,67.901610
request_accept_encoding,541515,59.650369
request_accept,526032,57.944846
request_cookie,462553,50.952342
request_connection,424955,46.810749


## 5. CAPEC Multi-Label Attack Distribution

We inspect the prevalence of each attack category in the dataset to assess class imbalance.

In [7]:
label_counts = df[ALL_LABEL_COLUMNS].sum().sort_values(ascending=False)
label_df = pd.DataFrame({
    "CAPEC Attack Label": label_counts.index,
    "Positive Samples": label_counts.values,
    "Prevalence (%)": (label_counts.values / len(df)) * 100
})

display(label_df)

# Check the single-sample rare class
rare_count = int(label_counts[RARE_LABEL])
print(f"\nNotice: '{RARE_LABEL}' has only {rare_count} positive sample in the entire dataset.")
print("Academic Decision: It is impossible to evaluate a class with N=1 in both train and test splits.")
print(f"Therefore, it is excluded from supervised model training, leaving {len(MODEL_LABEL_COLUMNS)} attack labels.")

,CAPEC Attack Label,Positive Samples,Prevalence (%)
0,000 - Normal,525195,57.852646
1,66 - SQL Injection,250311,27.572909
2,194 - Fake the Source of Data,56145,6.184630
3,126 - Path Traversal,20992,2.312365
4,34 - HTTP Response Splitting,19738,2.174232
5,242 - Code Injection,15827,1.743417
6,272 - Protocol Manipulation,9153,1.008245
7,88 - OS Command Injection,7482,0.824177
8,274 - HTTP Verb Tampering,5437,0.598911
9,310 - Scanning for Vulnerable Software,2718,0.299400



Notice: '248 - Command Injection' has only 1 positive sample in the entire dataset.
Academic Decision: It is impossible to evaluate a class with N=1 in both train and test splits.
Therefore, it is excluded from supervised model training, leaving 13 attack labels.


## 6. Multi-Label Complexity

A crucial contribution of your research is that web requests can have multiple attack categories simultaneously. We analyze how many labels co-occur per request.

In [8]:
active_labels_per_request = df[ALL_LABEL_COLUMNS].sum(axis=1)
multi_summary = active_labels_per_request.value_counts().sort_index()

for n_labels, count in multi_summary.items():
    print(f"Requests with {n_labels} active attack labels: {count:,} ({(count/len(df))*100:.2f}%)")

Requests with 1 active attack labels: 898,577 (98.98%)
Requests with 2 active attack labels: 8,132 (0.90%)
Requests with 3 active attack labels: 1,088 (0.12%)
Requests with 4 active attack labels: 18 (0.00%)


## 7. Conclusions and Next Steps for Phase 2

1. **Cleaning Rules Confirmed:**
   - Remove exact duplicate rows (2,746).
   - Remove 1 incomplete core row (index 14410).
   - Resulting clean dataset size: **905,068 records**.
2. **Feature Separation Confirmed:**
   - 14 request-side features selected for ML detection.
   - 5 contextual features preserved separately for PP2 correlation.
   - Response-side attributes and `request_do_not_track` excluded.
3. **Target Labels Confirmed:**
   - 13 attack labels retained for supervised detection (`248 - Command Injection` excluded due to $N=1$).
4. **Next Phase:** Proceed to Phase 2 (Data Preparation, Train/Test Split, and Context Preservation).